## Variance evaluations 
- Can we decrease the score variance of the outputs, making the outputs more consistant?

- H0: There is no statistically significant change in variance. 
- H1: There is a statistically significant change in variance.
- We will comparing per-sample std devs (or ranges) instead of means using a paired Wilcoxon signed-rank test on the per-sample std devs (5 pairs).




In [41]:
import pandas as pd
from scipy.stats import t
import numpy as np

## Pilot evaluation using n=15
- 5 videos x 3 runs
- less expensive, should so some useful data

#### Test 1: baseline

In [18]:
test_1 = pd.read_csv("../llm-evals/variance_test_data/test01-variance-test-baseline.csv")
test_1.head()


,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,1356d8bf-56fd-4340-a5c6-33a7bc113ae8,"{""user_query"": ""How's my form?""}","{""answer"": ""Bar path shifts forward and backwa...",variance-test-baseline-a6ffec26,1,"{""answer"": ""Overall: solid start. You’re contr...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,16.169876,9786,0.015015,0.5,1.0
1,1356d8bf-56fd-4340-a5c6-33a7bc113ae8,"{""user_query"": ""How's my form?""}","{""answer"": ""Bar path shifts forward and backwa...",variance-test-baseline-a6ffec26,2,"{""answer"": ""Overall: solid beginner-to-interme...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,13.403911,9430,0.025634,0.0,1.0
2,1356d8bf-56fd-4340-a5c6-33a7bc113ae8,"{""user_query"": ""How's my form?""}","{""answer"": ""Bar path shifts forward and backwa...",variance-test-baseline-a6ffec26,3,"{""answer"": ""Overall: solid start. The reps loo...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,12.975870,9398,0.025352,0.5,1.0
3,17c63e2c-303e-4e78-a817-a80eb3d6a9cf,"{""user_query"": ""How do I look?""}","{""answer"": ""Elbows flare a bit on the descent ...",variance-test-baseline-a6ffec26,1,"{""answer"": ""You look solid overall—controlled ...","{""inputs"": {""user_query"": ""How do I look?""}, ""...",success,NaN,14.441416,9279,0.021471,1.0,1.0
4,17c63e2c-303e-4e78-a817-a80eb3d6a9cf,"{""user_query"": ""How do I look?""}","{""answer"": ""Elbows flare a bit on the descent ...",variance-test-baseline-a6ffec26,2,"{""answer"": ""You look solid overall — the reps ...","{""inputs"": {""user_query"": ""How do I look?""}, ""...",success,NaN,20.057334,9284,0.025675,1.0,1.0


In [23]:
baseline_correctness_range = test_1.groupby("id")["correctness"].agg(lambda x: x.max() - x.min())
baseline_correctness_range
# .agg(...) = "aggregate each group into a single value"
# lambda x: x.max() - x.min() = "for each group (x), return max minus min"

id
1356d8bf-56fd-4340-a5c6-33a7bc113ae8    0.5
17c63e2c-303e-4e78-a817-a80eb3d6a9cf    0.5
2f0e4c5c-b788-46d9-9103-136294ad1e5a    0.5
402dde3c-02e5-46b6-b11b-af2cbc8e7e16    0.0
6a3dbbf9-89da-41e2-a6d8-c067a9a91a5f    0.5
Name: correctness, dtype: float64

In [24]:
baseline_hallucination_range = test_1.groupby("id")["hallucination"].agg(lambda x: x.max() - x.min())
baseline_hallucination_range

id
1356d8bf-56fd-4340-a5c6-33a7bc113ae8    0.0
17c63e2c-303e-4e78-a817-a80eb3d6a9cf    0.0
2f0e4c5c-b788-46d9-9103-136294ad1e5a    0.5
402dde3c-02e5-46b6-b11b-af2cbc8e7e16    0.0
6a3dbbf9-89da-41e2-a6d8-c067a9a91a5f    0.0
Name: hallucination, dtype: float64

### Find the Margin of Error for n=15

In [58]:
# With n=15, we will use a t-distribution value instead of normal distrobution
# Why: The normal distribution assumes you have enough data to know the true population std dev. 
# When n is small, you're estimating std dev from a tiny sample, which adds extra uncertainty on top of the regular sampling uncertainty.
# All 15 correctness scores from baseline
scores = test_1["correctness"]
n = len(scores)
mean = scores.mean()
std = scores.std()
moe = t.ppf(0.975, df=n-1) * (std / np.sqrt(n)) # t-value at 95% confidence is 2.14

print(f"Baseline (n={n}): mean = {mean:.2f} ± {moe:.2f}")

Baseline (n=15): mean = 0.70 ± 0.18


In [46]:
scores = test_1["hallucination"]
n = len(scores)
mean = scores.mean()
std = scores.std()
moe = t.ppf(0.975, df=n-1) * (std / np.sqrt(n))

print(f"Baseline (n={n}): mean = {mean:.2f} ± {moe:.2f}")

Baseline (n=15): mean = 0.93 ± 0.10


#### Test 2: router temp change to 0.0

In [16]:
test_2 = pd.read_csv("../llm-evals/variance_test_data/test02-variance-test-router-temp-0.0.csv")
test_2.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,1356d8bf-56fd-4340-a5c6-33a7bc113ae8,"{""user_query"": ""How's my form?""}","{""answer"": ""Bar path shifts forward and backwa...",variance-test-router-temp-0.0-c477216f,1,"{""answer"": ""Pretty solid base — you’re control...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,14.870959,9611,0.022922,1.0,1.0
1,1356d8bf-56fd-4340-a5c6-33a7bc113ae8,"{""user_query"": ""How's my form?""}","{""answer"": ""Bar path shifts forward and backwa...",variance-test-router-temp-0.0-c477216f,2,"{""answer"": ""Overall: solid, controlled benchin...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,14.087562,9396,0.021381,0.5,1.0
2,1356d8bf-56fd-4340-a5c6-33a7bc113ae8,"{""user_query"": ""How's my form?""}","{""answer"": ""Bar path shifts forward and backwa...",variance-test-router-temp-0.0-c477216f,3,"{""answer"": ""Overall: solid start, and the reps...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,14.498756,9323,0.021755,1.0,0.5
3,17c63e2c-303e-4e78-a817-a80eb3d6a9cf,"{""user_query"": ""How do I look?""}","{""answer"": ""Elbows flare a bit on the descent ...",variance-test-router-temp-0.0-c477216f,1,"{""answer"": ""You look solid overall — controlle...","{""inputs"": {""user_query"": ""How do I look?""}, ""...",success,NaN,16.343440,9366,0.026086,0.5,1.0
4,17c63e2c-303e-4e78-a817-a80eb3d6a9cf,"{""user_query"": ""How do I look?""}","{""answer"": ""Elbows flare a bit on the descent ...",variance-test-router-temp-0.0-c477216f,2,"{""answer"": ""You look solid overall — controlle...","{""inputs"": {""user_query"": ""How do I look?""}, ""...",success,NaN,14.123993,9266,0.025221,0.5,0.5


In [25]:
temp0_correctness_range = test_2.groupby("id")["correctness"].agg(lambda x: x.max() - x.min())
temp0_correctness_range

id
1356d8bf-56fd-4340-a5c6-33a7bc113ae8    0.5
17c63e2c-303e-4e78-a817-a80eb3d6a9cf    0.0
2f0e4c5c-b788-46d9-9103-136294ad1e5a    0.5
402dde3c-02e5-46b6-b11b-af2cbc8e7e16    0.5
6a3dbbf9-89da-41e2-a6d8-c067a9a91a5f    0.5
Name: correctness, dtype: float64

In [27]:
temp0_hallucination_range = test_2.groupby("id")["hallucination"].agg(lambda x: x.max() - x.min())
temp0_hallucination_range

id
1356d8bf-56fd-4340-a5c6-33a7bc113ae8    0.5
17c63e2c-303e-4e78-a817-a80eb3d6a9cf    0.5
2f0e4c5c-b788-46d9-9103-136294ad1e5a    0.5
402dde3c-02e5-46b6-b11b-af2cbc8e7e16    0.0
6a3dbbf9-89da-41e2-a6d8-c067a9a91a5f    0.0
Name: hallucination, dtype: float64

In [47]:
scores = test_2["correctness"]
n = len(scores)
mean = scores.mean()
std = scores.std()
moe = t.ppf(0.975, df=n-1) * (std / np.sqrt(n))

print(f"Baseline (n={n}): mean = {mean:.2f} ± {moe:.2f}")

Baseline (n=15): mean = 0.73 ± 0.14


In [48]:
scores = test_2["hallucination"]
n = len(scores)
mean = scores.mean()
std = scores.std()
moe = t.ppf(0.975, df=n-1) * (std / np.sqrt(n))

print(f"Baseline (n={n}): mean = {mean:.2f} ± {moe:.2f}")

Baseline (n=15): mean = 0.87 ± 0.13


In [28]:
# make a dataframe called "comparison" using the baseline range and new range for each score type
# add a column to that dataframe called "delta", which = the delta between new range - baseline range

comparison = pd.DataFrame({"baseline_hallucination_range": baseline_hallucination_range, "temp0_hallucination_range": temp0_hallucination_range,
                           "baseline_correctness_range": baseline_correctness_range, "temp0_correctness_range": temp0_correctness_range})
comparison["hallucination_delta"] = comparison["temp0_hallucination_range"] - comparison["baseline_hallucination_range"]
comparison["correctness_delta"] = comparison["temp0_correctness_range"] - comparison["baseline_correctness_range"]
comparison

,baseline_hallucination_range,temp0_hallucination_range,baseline_correctness_range,temp0_correctness_range,hallucination_delta,correctness_delta
id,,,,,,
1356d8bf-56fd-4340-a5c6-33a7bc113ae8,0.0,0.5,0.5,0.5,0.5,0.0
17c63e2c-303e-4e78-a817-a80eb3d6a9cf,0.0,0.5,0.5,0.0,0.5,-0.5
2f0e4c5c-b788-46d9-9103-136294ad1e5a,0.5,0.5,0.5,0.5,0.0,0.0
402dde3c-02e5-46b6-b11b-af2cbc8e7e16,0.0,0.0,0.0,0.5,0.0,0.5
6a3dbbf9-89da-41e2-a6d8-c067a9a91a5f,0.0,0.0,0.5,0.5,0.0,0.0


In [ ]:
# delta = temp0 - baseline: if detla is positive, temp0 variance is bigger
# Tempature intervention mostly did nothing, and where it did something, it was slightly negative 
# (2 videos got worse on hallucination variance, 1 got worse on correctness, 1 got better on correctness). 
# Net: noise, or a very slight negative effect. Not a win.

#### Test 3: scaffold update

In [54]:
test_3 = pd.read_csv("../llm-evals/variance_test_data/test03-system-prompt-scaffold-update.csv")

test_3_correctness_range = test_3.groupby("id")["correctness"].agg(lambda x: x.max() - x.min())
test_3_hallucination_range = test_3.groupby("id")["hallucination"].agg(lambda x: x.max() - x.min())

In [53]:
scores = test_3["correctness"]
n = len(scores)
mean = scores.mean()
std = scores.std()
moe = t.ppf(0.975, df=n-1) * (std / np.sqrt(n))

print(f"Baseline (n={n}): mean = {mean:.2f} ± {moe:.2f}")

Baseline (n=15): mean = 0.63 ± 0.16


In [56]:
scores = test_2["hallucination"]
n = len(scores)
mean = scores.mean()
std = scores.std()
moe = t.ppf(0.975, df=n-1) * (std / np.sqrt(n))

print(f"Baseline (n={n}): mean = {mean:.2f} ± {moe:.2f}")

Baseline (n=15): mean = 0.87 ± 0.13


In [55]:
comparison = pd.DataFrame({"baseline_hallucination_range": baseline_hallucination_range, "test_3_hallucination_range":test_3_hallucination_range,
                           "baseline_correctness_range": baseline_correctness_range, "test_3_correctness_range": test_3_correctness_range})
comparison["hallucination_delta"] = comparison["test_3_hallucination_range"] - comparison["baseline_hallucination_range"]
comparison["correctness_delta"] = comparison["test_3_correctness_range"] - comparison["baseline_correctness_range"]
comparison

,baseline_hallucination_range,test_3_hallucination_range,baseline_correctness_range,test_3_correctness_range,hallucination_delta,correctness_delta
id,,,,,,
1356d8bf-56fd-4340-a5c6-33a7bc113ae8,0.0,0.0,0.5,0.0,0.0,-0.5
17c63e2c-303e-4e78-a817-a80eb3d6a9cf,0.0,0.5,0.5,0.0,0.5,-0.5
2f0e4c5c-b788-46d9-9103-136294ad1e5a,0.5,0.5,0.5,0.5,0.0,0.0
402dde3c-02e5-46b6-b11b-af2cbc8e7e16,0.0,0.5,0.0,1.0,0.5,1.0
6a3dbbf9-89da-41e2-a6d8-c067a9a91a5f,0.0,0.0,0.5,0.0,0.0,-0.5


In [59]:
from scipy.stats import ttest_rel

baseline_means = test_1.groupby("id")["correctness"].mean()
scaffold_means = test_3.groupby("id")["correctness"].mean()

t_stat, p_value = ttest_rel(baseline_means, scaffold_means)
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.3f}")

Paired t-test: t = 0.535, p = 0.621


In [60]:
from scipy.stats import wilcoxon

baseline_ranges = test_1.groupby("id")["correctness"].agg(lambda x: x.max() - x.min())
scaffold_ranges = test_3.groupby("id")["correctness"].agg(lambda x: x.max() - x.min())

stat, p_value = wilcoxon(baseline_ranges, scaffold_ranges)
print(f"Wilcoxon: stat = {stat:.3f}, p = {p_value:.3f}")

Wilcoxon: stat = 4.000, p = 1.000
